# Notebook 18 — Build a RAG System on a Real Filing
**AI Models and Agents for Business · Week 9 · Session 18**

In Notebook 17 you saw how embeddings measure meaning. Today you put the pieces together on a long, real document, Apple's annual report (10-K) for fiscal 2023, and run two experiments that change the answers. These are the same settings your team will tune in Project 2.

| Cells | What you do |
|---|---|
| A1-A6 | Load, chunk, embed and search the 10-K |
| A7-A8 | Ask with retrieval, then from memory only |
| A9 | Score five answers: correct, cited, declined when it should |
| A10 | Experiment 1: chunk size |
| A11 | Experiment 2: how many passages to send |

---
## Build a RAG Pipeline on a Real PDF

### What is RAG?
**Retrieval-Augmented Generation** grounds a language model's answers in a
specific document, rather than relying on what it memorized during training.

**Without RAG:** the model guesses from training data -> possible hallucination, no citation possible.
**With RAG:** relevant paragraphs are retrieved from the actual document -> an answer that *can* be grounded and cited.

RAG makes a grounded answer possible; it doesn't guarantee one. Retrieval can miss the right passage, and the model can still misread, mix up or misattribute the passages it was given. Today you'll see both kinds of failure.

### Architecture (the most common design; some RAG systems retrieve with keyword search or a database query instead of embeddings)
```
PDF  ->  [Chunk]  ->  [Embed]  ->  [Store]
                                     |
Question -> [Embed] -> [Retrieve top-k chunks] -> [Generate answer] -> Answer + Citations
```

### What's free in this part
- **Embeddings:** `sentence-transformers` runs locally — no API key, no cost
- **Generation:** Groq (free tier, openai/gpt-oss-20b)

In [ ]:
# Cell A1 — Install libraries (takes ~60 seconds)
!pip install groq sentence-transformers pypdf numpy --quiet

import numpy as np
import textwrap
import urllib.request
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from groq import Groq
import warnings
warnings.filterwarnings('ignore')
print("Libraries ready.")

In [ ]:
# Cell A2 — API key and models
# Your Groq key comes from Colab Secrets (key icon on the left, name GROQ_API_KEY, Notebook access on).
# If it is not there, you will be asked to paste it in a box that hides what you type.
def _get_key():
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        import getpass
        return getpass.getpass("Paste your Groq API key (starts with gsk_): ")

GROQ_API_KEY = _get_key()
groq_client = Groq(api_key=GROQ_API_KEY)

# CHANGE ME — try "openai/gpt-oss-120b" for better answers (slower; uses more of the free daily limit)
LLM_MODEL = "openai/gpt-oss-20b"

# Groq retired llama-3.1-8b-instant for free accounts in August 2026, so this notebook uses
# openai/gpt-oss-20b. It is a reasoning model: before it answers, it writes hidden "thinking"
# tokens. They count toward max_tokens and are billed as output tokens. The lines below keep
# that thinking short and add room for it, so every max_tokens in this notebook still means
# "room for the answer".
REASONING_EFFORT = "low"      # "low", "medium" or "high"
REASONING_HEADROOM = 300      # extra tokens reserved for the hidden thinking
_groq_create = groq_client.chat.completions.create
def _create_with_reasoning(*args, **kwargs):
    if "gpt-oss" in str(kwargs.get("model", "")):
        kwargs.setdefault("reasoning_effort", REASONING_EFFORT)
        kwargs.setdefault("include_reasoning", False)
        if kwargs.get("max_tokens"):
            kwargs["max_tokens"] += REASONING_HEADROOM
    return _groq_create(*args, **kwargs)
groq_client.chat.completions.create = _create_with_reasoning

# The embedding model runs LOCALLY — no API key needed
print("Loading embedding model (~30 seconds first time)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model ready.")
print(f"Groq LLM ready: {LLM_MODEL}")
print()
print("Note: embeddings are free and run on Colab's CPU.")
print("Only the final answer-generation step uses the Groq API.")

In [ ]:
# Cell A3 — Load the PDF
# We use Apple's 2023 Annual Report (10-K), publicly available.
# CHANGE ME — swap for any publicly accessible PDF URL
PDF_URL  = "https://s2.q4cdn.com/470004039/files/doc_earnings/2023/ar/_10-k-2023-(as-filed).pdf"
PDF_PATH = "annual_report.pdf"

import os
print("Downloading PDF...")
try:
    req = urllib.request.Request(PDF_URL, headers={"User-Agent": "Mozilla/5.0"})  # some sites refuse scripts without a browser name
    with urllib.request.urlopen(req, timeout=60) as resp, open(PDF_PATH, "wb") as out:
        out.write(resp.read())
except Exception as err:
    if not os.path.exists(PDF_PATH):
        raise SystemExit(f"Download failed ({err}). Upload the PDF yourself: Files panel (folder icon on the left) "
                         f"-> Upload, name it {PDF_PATH}, then re-run this cell.")
    print(f"Download failed ({err}); using the copy of {PDF_PATH} already in Files.")

reader = PdfReader(PDF_PATH)
pages  = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text and len(text.strip()) > 100:
        pages.append({"page": i + 1, "text": text.strip()})

total_chars = sum(len(p["text"]) for p in pages)
print("PDF loaded.")
print(f"   Pages with content: {len(pages)}")
print(f"   Total characters:   {total_chars:,}")
print(f"   Approx. tokens:     {total_chars//4:,}")
print()
print("Sample from page 1:")
print(pages[0]["text"][:300] + "...")

In [ ]:
# Cell A4 — Chunk the document
# CHANGE ME — try 200, 400, 800. Compare in Cell A10.
CHUNK_SIZE    = 400   # characters per chunk
CHUNK_OVERLAP = 80    # overlap between consecutive chunks

def chunk_text(pages, chunk_size, overlap):
    chunks = []
    for p in pages:
        text = p["text"]
        start = 0
        while start < len(text):
            chunk = text[start:start + chunk_size].strip()
            if len(chunk) > 50:
                chunks.append({"text": chunk, "page": p["page"]})
            start += chunk_size - overlap
    return chunks

chunks = chunk_text(pages, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"Chunk size:    {CHUNK_SIZE} chars  |  Overlap: {CHUNK_OVERLAP} chars")
print(f"Total chunks:  {len(chunks):,}")
print(f"Avg chunk:     {sum(len(c['text']) for c in chunks)//len(chunks)} chars")
print()
print("Example chunk:")
print("-"*50)
print(f"Page {chunks[10]['page']}:  {chunks[10]['text']}")

In [ ]:
# Cell A5 — Embed all chunks (1-3 minutes)
print(f"Embedding {len(chunks):,} chunks on Colab CPU...")
print("(Progress bar below — takes 1-3 minutes)")

texts      = [c["text"] for c in chunks]
embeddings = embedder.encode(texts, show_progress_bar=True, batch_size=64)

print()
print("Done.")
print(f"   Embedding matrix: {embeddings.shape}  ({embeddings.shape[0]:,} chunks x {embeddings.shape[1]} dimensions)")
print()
print("What just happened:")
print("  Each chunk of text -> a vector of 384 numbers")
print("  Similar chunks end up close together in this 384-dimensional space")
print("  This is what makes semantic search possible — no keywords needed")

In [ ]:
# Cell A6 — Retrieval function
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(question, top_k=3):
    q_emb  = embedder.encode([question])[0]
    scores = [cosine_similarity(q_emb, emb) for emb in embeddings]
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [{"text": chunks[i]["text"], "page": chunks[i]["page"],
             "similarity": scores[i]} for i in top_idx]

# Quick test
test_q  = "What was Apple's total revenue?"
results = retrieve(test_q, top_k=2)
print(f"Test retrieval: '{test_q}'")
for i, r in enumerate(results, 1):
    print(f"\n  Result {i} | Page {r['page']} | Similarity: {r['similarity']:.3f}")
    print(f"  {r['text'][:200]}...")

In [ ]:
# Cell A7 — Ask questions WITH RAG
def ask_with_rag(question, top_k=3, show_sources=True):
    retrieved = retrieve(question, top_k=top_k)

    context = ""
    for i, r in enumerate(retrieved, 1):
        context += f"[Source {i} — Page {r['page']}]\n{r['text']}\n\n"

    system = """You are a financial analyst assistant.
Answer questions using ONLY the provided source excerpts.
Always cite which source you draw from (e.g. 'According to Source 2, page 47...').
If the sources don't contain enough information, say so explicitly.
Do not use any knowledge from outside the provided sources."""

    prompt = f"""Answer this question using only the sources below:

QUESTION: {question}

SOURCES:
{context}

Answer:"""

    r = groq_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role":"system","content":system},
                  {"role":"user","content":prompt}],
        max_tokens=400, temperature=0.1
    )
    answer = r.choices[0].message.content

    print(f"QUESTION: {question}")
    if show_sources:
        print("\nRETRIEVED SOURCES:")
        for i, s in enumerate(retrieved, 1):
            print(f"  Source {i} (p.{s['page']}, sim={s['similarity']:.3f}): {s['text'][:100]}...")
    print("\nANSWER (with RAG):")
    print(answer)
    return answer

# CHANGE ME — ask anything about the annual report
answer1 = ask_with_rag("What were Apple's total net sales for fiscal year 2023?")

In [ ]:
# Cell A8 — Same question WITHOUT RAG <- KEY EXPERIMENT
def ask_without_rag(question):
    r = groq_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role":"user","content":question}],
        max_tokens=300, temperature=0.1
    )
    return r.choices[0].message.content

# CHANGE ME — use the same question as Cell A7
question      = "What were Apple's total net sales for fiscal year 2023?"
no_rag_answer = ask_without_rag(question)

print("SAME QUESTION — NO RAG (model relies on training data only)")
print("=" * 60)
print(f"Q: {question}")
print()
print("ANSWER:")
print(no_rag_answer)
print()
print("=" * 60)
print("Compare to the RAG answer above.")
print("  1. Does the no-RAG answer cite a source you can verify?")
print("  2. How would you know if it's accurate?")
print("  3. What if the training data predates this filing?")

In [ ]:
# Cell A9 — Test a variety of questions
# CHANGE ME — mix specific (should work) and vague or future (should fail gracefully)
questions = [
    "What are the main risk factors Apple disclosed?",
    "How many employees does Apple have?",
    "What did Tim Cook say about artificial intelligence?",
    "What is Apple's strategy for India?",
    "What will Apple's revenue be next year?",   # should fail gracefully
]

for q in questions:
    print("\n" + "=" * 65)
    _ = ask_with_rag(q, top_k=3, show_sources=False)

### Score the five answers (pairs)
For each answer above, mark three things. This is Project 2's scoring, on a smaller scale.

| Question | Correct? (check the filing) | Cited a page that supports it? | Declined when it should? |
|---|---|---|---|
| Risk factors | | | |
| Employees | | | |
| Tim Cook on AI | | | |
| India strategy | | | |
| Next year's revenue | | | |

In [ ]:
# Cell A10 — Chunk size experiment <- KEY EXPERIMENT
# CHANGE ME — compare to the CHUNK_SIZE in Cell A4 (try 200, 400, 800)
EXP_CHUNK_SIZE = 200

print(f"Re-chunking with chunk size = {EXP_CHUNK_SIZE}...")
chunks_exp     = chunk_text(pages, EXP_CHUNK_SIZE, overlap=40)
texts_exp      = [c["text"] for c in chunks_exp]
embeddings_exp = embedder.encode(texts_exp, show_progress_bar=True, batch_size=64)
print(f"Original: {len(chunks):,} chunks  |  Experiment: {len(chunks_exp):,} chunks")

def retrieve_exp(question, top_k=3):
    q_emb  = embedder.encode([question])[0]
    scores = [cosine_similarity(q_emb, emb) for emb in embeddings_exp]
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [{"text": chunks_exp[i]["text"], "page": chunks_exp[i]["page"]} for i in top_idx]

test_q   = "What were Apple's total net sales for fiscal year 2023?"
ctx_orig = "".join(f"[Source {i+1} p.{r['page']}]\n{r['text']}\n\n"
                   for i, r in enumerate(retrieve(test_q)))
ctx_exp  = "".join(f"[Source {i+1} p.{r['page']}]\n{r['text']}\n\n"
                   for i, r in enumerate(retrieve_exp(test_q)))

def gen(context, question):
    r = groq_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role":"system","content":"Answer using ONLY the sources. Cite pages. Say if unsure."},
                  {"role":"user","content":f"Q: {question}\n\nSources:\n{context}\nAnswer:"}],
        max_tokens=250, temperature=0.1
    )
    return r.choices[0].message.content

print(f"\nQ: {test_q}")
print(f"\n-- Chunk size {CHUNK_SIZE} (original) --")
print(gen(ctx_orig, test_q))
print(f"\n-- Chunk size {EXP_CHUNK_SIZE} (experiment) --")
print(gen(ctx_exp, test_q))
print()
print("Trade-off to discuss:")
print(f"  Smaller chunks ({EXP_CHUNK_SIZE}): more precise retrieval, less context per chunk")
print(f"  Larger chunks  ({CHUNK_SIZE}): more context, but retrieval is less targeted")

---
## Experiment 2 — How many passages to send?

`retrieve()` returns the `top_k` most similar chunks (3 by default). More passages raise the chance the answer is in there, but cost more input tokens and can bury the answer; too few and the answer may simply be missing.

**Predict first:** for each question, which `top_k` will give the best answer?

In [ ]:
# Cell A11 — How many passages to send? <- KEY EXPERIMENT
# CHANGE ME — try other values of top_k, or your own questions
TOP_K_VALUES  = [1, 3, 8]
EXP_QUESTIONS = [
    "What were Apple's total net sales for fiscal year 2023?",
    "Why did Apple's gross margin change in fiscal 2023?",
]
PRICE_IN = 0.075   # $ per million input tokens, openai/gpt-oss-20b on Groq (checked October 2026)

import time, groq

def answer_with_k(question, k):
    retrieved = retrieve(question, top_k=k)
    context = "".join(f"[Source {i} — Page {r['page']}]\n{r['text']}\n\n" for i, r in enumerate(retrieved, 1))
    for attempt in range(6):
        try:
            r = groq_client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "system", "content": "Answer using ONLY the sources. Cite pages. "
                                                        "Say so if the sources do not contain the answer."},
                          {"role": "user", "content": f"Q: {question}\n\nSources:\n{context}\nAnswer:"}],
                max_tokens=250, temperature=0.1)
            return r.choices[0].message.content, r.usage.prompt_tokens, sorted({x["page"] for x in retrieved})
        except groq.RateLimitError:
            print("(rate limit hit - waiting 20 seconds, then trying again)")
            time.sleep(20)
    raise RuntimeError("Still rate limited. Wait a minute and re-run this cell.")

rows = []
for q in EXP_QUESTIONS:
    print("=" * 70)
    print("Q:", q)
    for k in TOP_K_VALUES:
        answer, tokens_in, pages_used = answer_with_k(q, k)
        rows.append({"question": q[:45] + "...", "top_k": k, "input tokens": tokens_in,
                     "input cost per million questions ($)": round(tokens_in * PRICE_IN, 2)})
        print(f"\n-- top_k = {k} | pages sent: {pages_used} | input tokens: {tokens_in:,}")
        print(textwrap.fill(answer, 100))

import pandas as pd
display(pd.DataFrame(rows))

💬 **Discuss:** Did more passages make either answer better? Worse? The money is small per question; at a million questions a month, what does the extra input cost? Which is usually the bigger cost of sending too many passages: the bill, or accuracy?

### When an answer is wrong, find out where
- **Retrieval error:** the right passage never reached the model (check the pages it was sent). Fixes: chunk size, more passages, different wording, hybrid search.
- **Generation error:** the right passage was there and the model still got it wrong. Fixes: a clearer instruction, a requirement to quote or cite, a stronger model.

### ✅ Concept check — RAG
Answer each one in your head, or out loud with your partner, *before* you open the answer. If you miss one, re-read that part before moving on.

**1. The assistant gives a wrong answer. How do you tell a retrieval error from a generation error?**
<details><summary>Check your answer</summary>

Look at the passages it was **sent**. If the right passage isn't among them, it's a retrieval error. If it is there and the answer is still wrong, it's a generation error.
</details>

**2. The answer cites page 23. Does that prove the answer is right?**
<details><summary>Check your answer</summary>

No. A model can cite a page that doesn't support its claim, or misread a page it did cite. Check the cited passage yourself; that is what the "Cited a page that supports it?" column is for.
</details>

**3. You raise `top_k` from 3 to 10. Give one benefit and two costs.**
<details><summary>Check your answer</summary>

Benefit: a better chance the right passage is included. Costs: more input tokens on every question, and the answer can be **buried** among irrelevant passages, which can make generation worse.
</details>

---
## ✅ Notebook 18 complete — Reflection (submit with your notebook)

These prep you for **Quiz 5** (Session 20).

1. Why does RAG reduce hallucination compared to a base LLM answering from memory?
2. What is an embedding, and how does cosine similarity enable semantic search without keywords?
3. What happens when the answer is spread across multiple pages and no single chunk contains the full picture?
4. In Cell A11, what happened to answer quality and to input tokens as you sent more passages? Which `top_k` would you start with for Project 2, and why?
5. Pick one wrong answer you saw today. Was it a retrieval error or a generation error? How can you tell?

*Next session: Skills Files, AGENTS.md and Workflows (Notebook 19).*